In [ ]:
"""
2Nadkarni et al., “Natural language processing: an introduction”. JAMIA https://www.ncbi.nlm.nih.gov/pmc/articles/
PMC3168328
3Wikipedia entry for natural language processing: https://en.wikipedia.org/wiki/Natural-language_processing

Your goal in this chapter is to turn text into something that a neural network can
process, which, like the previous cases, is a tensor of numbers. If you can do that and
later choose the right architecture for your text processing job, you’ll be in the position
of doing NLP with PyTorch. You see right away how powerful this capability is:
you can achieve state-of-the-art performance on tasks in different domains with the
same PyTorch tools if you cast your problem in the right form. The first part of this job is
reshaping data.

Networks operate on text at two levels: at character level, by processing one character
at a time, and at word level, in which individual words are the finest-grained entities
seen by the network. The technique you use to encode text information into
tensor form is the same whether you operate at character level or at word level. This
technique is nothing magic; you stumbled upon it earlier. It’s one-hot encoding

Start with a character-level example. First, get some text to process. An amazing
resource is Project Gutenberg4, a volunteer effort that digitizes and archives cultural
work and makes it available for free in open formats, including plain-text files.
If you’re aiming at larger-scale corpora, the Wikipedia corpus stands out: it’s the
complete collection of Wikipedia articles containing 1.9 billion words and more
than 4.4 million articles. You can find several other corpora at the English Corpora
website.5
Load Jane Austen’s Pride and Prejudice from the Project Gutenberg website.6 Save
the file and read it in, as shown in the following listing.

4 http://www.gutenberg.org
5 https://www.english-corpora.org
6 http://www.gutenberg.org/files/1342/1342-0.txt
"""

In [23]:
import torch 

with open('1342-0.txt', encoding='utf8') as f:
    text = f.read()

In [ ]:
"""
You need to take care of one more detail before you proceed: encoding. Encoding is a
vast subject, so all we’ll do now is touch on it. Every written character is represented by
a code, a sequence of bits of appropriate length that allow each character to be
uniquely identified. The simplest such encoding is ASCII (American Standard Code
for Information Interchange), dating back to the 1960s. ASCII encodes 128 characters
using 128 integers. Letter a, for example, corresponds to binary 1100001 or decimal
97; letter b corresponds to binary 1100010 or decimal 98, and so on. The encoding
would fit 8 bits, which was a big bonus in 1965.

NOTE Clearly, 128 characters aren’t enough to account for all the glyphs,
accents, ligatures, and other features that are needed to properly represent
written text in languages other than English. To this end, other encodings
have been developed, using a larger number of bits as a code for a wider
range of characters. That wider range of characters got standardized as Unicode,
which maps all known characters to numbers, with the representation in
bits of those numbers being provided by a specific encoding. Popular encodings
include UTF-8, UTF-16 and UTF-32, in which the numbers are a sequence
of 8-, 16-, or 32-bit integers. Strings in Python 3.x are Unicode strings.

You’re going to one-hot encode your characters to limit the one-hot encoding to a
character set that’s useful for the text being analyzed. In this case, because you loaded
text in English, it’s quite safe to use ASCII and deal with a small encoding. You could
also make all characters lowercase to reduce the number of characters in your encoding.
Similarly, you could screen out punctuation, numbers, and other characters that
aren’t relevant to the expected kinds of text, which may or may not make a practical
difference to your neural network, depending on the task at hand.
At this point, you need to parse the characters in the text and provide a one-hot encoding
for each of them. Each character will be represented by a vector of length equal to the
number of characters in the encoding. This vector will contain all zeros except for a 1 at
the index corresponding to the location of the character in the encoding.
First, split your text into a list of lines and pick an arbitrary line to focus on:
"""

In [25]:
lines = text.split('\n')
line = lines[200]
line

'for there was a distinctly feminine element in “Mr. Spectator,” and in'

In [ ]:
"""
Create a tensor that can hold the total number of one-hot encoded characters for the
whole line:
"""

In [26]:
letter_tensor = torch.zeros(len(line), 128)
letter_tensor.shape

torch.Size([70, 128])

In [ ]:
"""
Note that letter_tensor holds a one-hot encoded character per row. Now set a 1 on
each row in the right position so that each row represents the right character. The index
where the 1 has to be set corresponds to the index of the character in the encoding:
"""

In [27]:
for i, letter in enumerate(line.lower().strip()):
    letter_index = ord(letter) if ord(letter) < 128 else 0  # The text uses directional double quotes,
    # which aren’t valid ASCII, so screen them out here.
    letter_tensor[i] [letter_index] = 1

In [ ]:
"""
You’ve one-hot encoded your sentence into a representation that a neural network
can digest. You could do word-level encoding the same way by establishing a vocabulary
and one-hot encoding sentences, sequences of words, along the rows of your tensor.
Because a vocabulary contains many words, this method produces wide encoded
vectors that may not be practical. Later in this chapter, you see a more efficient way to
represent text at word level by using embeddings. For now, stick with one-hot encodings
to see what happens.
Define clean_words, which takes text and returns it lowercase and stripped of punctuation.
When you call it on your “Impossible, Mr. Bennet” line, you get the following:
"""

In [28]:
def clean_words (input_str):
    punctuation = '.,;:"!?”“_-'
    word_list = input_str.lower().replace('\n', ' ').split()
    word_list = [word.strip(punctuation) for word in word_list]
    return word_list

words_in_line = clean_words(line)
line, words_in_line

('for there was a distinctly feminine element in “Mr. Spectator,” and in',
 ['for',
  'there',
  'was',
  'a',
  'distinctly',
  'feminine',
  'element',
  'in',
  'mr',
  'spectator',
  'and',
  'in'])

In [ ]:
"""
Next, build a mapping of words to indexes in your encoding:
"""

In [29]:
word_list = sorted(set(clean_words(text)))
word2index_dict = {word: i for (i, word) in enumerate(word_list)}

len(word2index_dict), word2index_dict['impossible']


(7788, 3622)

In [ ]:
"""
Note that all_words is now a dictionary with words as keys and an integer as value.
You’ll use this dictionary to efficiently find the index of a word as you one-hot encode it.
Now focus on your sentence. Break it into words and one-hot encode it—that is,
populate a tensor with one one-hot encoded vector per word. Create an empty vector,
and assign the one-hot encoded values of the word in the sentence:
"""

In [37]:
word_tensor = torch.zeros(len(words_in_line), len(word2index_dict))
# print(word_tensor.shape)
for i, word in enumerate(words_in_line):
    word_index = word2index_dict[word]
    word_tensor[i][word_index] = 1
    print('{:2} {:4} {}'.format(i, word_index, word))

print(word_tensor.shape)

 0 2915 for
 1 6897 there
 2 7460 was
 3  193 a
 4 2180 distinctly
 5 2802 feminine
 6 2374 element
 7 3647 in
 8 4600 mr
 9 6495 spectator
10  497 and
11 3647 in
torch.Size([12, 7788])


In [ ]:
"""
At this point, tensor represents one sentence of length 12 in an encoding space of
size 7788—the number of words in your dictionary.

Text embeddings
One-hot encoding is a useful technique for representing categorical data in tensors.
As you may have anticipated, however, one-hot encoding starts to break down when
the number of items to encode is effectively unbound, as with words in a corpus. In
one book, you had more than 7,000 items!

You certainly could do some work to deduplicate words, condense alternative spellings,
collapse past and future tenses into a single token, and that kind of thing. Still, a
general-purpose English-language encoding is going to be huge. Worse, every time you
encounter a new word, you have to add a new column to the vector, which means adding
a new set of weights to the model to account for that new vocabulary entry, which
is going to be painful from a training perspective.

How can you compress your encoding to a more manageable size and put a cap on
the size growth? Well, instead of using vectors of many zeros and a single 1, you could
use vectors of floating-point numbers. A vector of, say, 100 floating-point numbers can
indeed represent a large number of words. The trick is to find an effective way to map
individual words to this 100-dimensional space in a way that facilitates downstream
learning. This technique is called embedding.

In principle, you could iterate over your vocabulary and generate a set of 100 random
floating-point numbers for each word. This method would work, in that you
could cram a large vocabulary into 100 numbers, but it would forgo any concept of
distance between words based on meaning or context. A model that used this word
embedding would have to deal with little structure in its input vectors. An ideal solution
would be to generate the embedding in such a way that words used in similar contexts
map to nearby regions of the embedding.

If you were to design a solution to this problem by hand, you might decide to
build your embedding space by mapping basic nouns and adjectives along the axes.
You can generate a 2D space in which axes map to nouns "fruit" (0.0–0.33),
"flower" (0.33–0.66), and "dog" (0.66–1.0), and to adjectives "red" (0.0–0.2),
"orange" (0.2–0.4), "yellow" (0.4–0.6), "white" (0.6–0.8), and "brown" (0.8–
1.0). Your goal now is to take actual fruit, flowers, and dogs and lay them out in the
embedding.

As you start embedding words, you can map "apple" to a number in the "fruit"
and "red" quadrant. Likewise, you can easily map "tangerine", "lemon", "lychee",
and "kiwi" (to round out your list of colorful fruits). Then you can start on flowers,
assigning "rose", "poppy", "daffodil", "lily", and . . . well, there aren’t many
brown flowers out there. Well, "sunflower" can get "flower", "yellow", and
"brown", and "daisy" can get "flower" "white", and "yellow". Perhaps you should
update "kiwi" to map close to "fruit", "brown", and "green". For dogs and color,
you can embed "redbone", "fox" perhaps for "orange", "golden retriever", "poodles"
for "white", and . . . most kinds of dogs are "brown".

Although doing this mapping manually isn’t feasible for a large corpus, you should
note that although you had an embedding size of 2, you described 15 different words
besides the base 8 and probably could cram quite a few more in if you take the time to be
creative.

As you’ve probably guessed, this kind of work can be automated. By processing a
large corpus of organic text, you can generate embeddings similar to this one. The
main differences are that the embedding vector has 100 to 1,000 elements and that
axes don’t map directly to concepts, but conceptually similar words map to neighboring
regions of an embedding space whose axes are arbitrary floating-point
dimensions.

Although the exact algorithms7 used are a bit out of scope for what we wanting to
focus on here, we’d like to mention that embeddings are often generated by using
neural networks, trying to predict a word from nearby words (the context) in a sentence.
In this case, you could start from one-hot encoded words and use a (usually
rather shallow) neural network to generate the embedding. When the embedding is
available, you could use it for downstream tasks.

7 One example is https://en.wikipedia.org/wiki/Word2vec

One interesting aspect of the resulting embeddings is that similar words end up
not only clustered together, but also with consistent spatial relationships with other
words. If you were to take the embedding vector for "apple" and begin to add and
subtract the vectors for other words, you could begin to perform analogies such as
apple - red - sweet + yellow + sour and end up with a vector similar to the one
for "lemon".

We won’t be using text embeddings here, but they’re essential tools when a large
number of entries in a set has to be represented with numeric vectors.
"""